# nanochat ROCm 完整训练流程

在 AMD MI300X GPU 上训练 nanochat 模型的完整流程

## 环境信息

- GPU: AMD MI300X (192GB HBM3)
- ROCm: 6.10.5
- PyTorch: 2.10.0 (ROCm)

In [ ]:
# 检查 GPU 环境
import subprocess
result = subprocess.run(['rocm-smi', '--showuse', '--showmeminfo', 'vram'], capture_output=True, text=True)
print(result.stdout[:500])

---
## 1. 克隆 nanochat 仓库

In [ ]:
import os
import subprocess

# 克隆仓库
if not os.path.exists('/mnt/workspace/nanochat'):
    result = subprocess.run(['git', 'clone', 'https://github.com/karpathy/nanochat.git', '/mnt/workspace/nanochat'], capture_output=True, text=True)
    print(result.stdout + result.stderr)
else:
    print('nanochat already cloned')

# 切换到目录
os.chdir('/mnt/workspace/nanochat')
print(f'Working directory: {os.getcwd()}')

---
## 2. 安装依赖

In [ ]:
# 安装必要的依赖
import subprocess

packages = ['wandb', 'rustbpe', 'huggingface_hub']
for pkg in packages:
    result = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'✓ {pkg} installed')
    else:
        print(f'✗ {pkg} failed: {result.stderr}')

---
## 3. 下载数据集

使用 HuggingFace 镜像下载数据

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from huggingface_hub import hf_hub_download
import subprocess

# 数据集信息
repo_id = 'karpathy/base_data_climbmix'
data_dir = os.path.expanduser('~/.cache/nanochat/base_data_climbmix/')

os.makedirs(data_dir, exist_ok=True)

# 下载 shards (155 个)
print('Downloading data shards...')
for i in range(155):
    shard_name = f'shard_{i:06d}.bin'
    shard_path = os.path.join(data_dir, shard_name)
    if not os.path.exists(shard_path):
        try:
            hf_hub_download(repo_id=repo_id, filename=shard_name, local_dir=data_dir)
            print(f'  ✓ {shard_name}')
        except Exception as e:
            print(f'  ✗ {shard_name}: {e}')
    else:
        print(f'  skip {shard_name} (exists)')

print(f'Data downloaded to: {data_dir}')

---
## 4. 训练 Tokenizer

In [ ]:
import subprocess
import os
os.chdir('/mnt/workspace/nanochat')

# 训练 tokenizer
result = subprocess.run([
    'python3', '-m', 'scripts.setup_pretrained_tokenizer',
    '--vocab-size', '32768'
], capture_output=True, text=True, timeout=300)

print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
print(result.stderr[-200:] if len(result.stderr) > 200 else result.stderr)

tokenizer_path = os.path.expanduser('~/.cache/nanochat/tokenizer/tokenizer.pkl')
print(f'Tokenizer saved to: {tokenizer_path}')

---
## 5. ROCm 环境适配

需要修改代码以适配 ROCm

In [ ]:
# 设置环境变量
import os
os.environ['NANOCHAT_DTYPE'] = 'float32'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['NANOCHAT_NO_COMPILE'] = '1'

print('Environment variables set:')
print(f'  NANOCHAT_DTYPE = {os.environ.get("NANOCHAT_DTYPE")}')
print(f'  WANDB_MODE = {os.environ.get("WANDB_MODE")}')
print(f'  NANOCHAT_NO_COMPILE = {os.environ.get("NANOCHAT_NO_COMPILE")}')

### 5.1 替换 Muon Optimizer 为 AdamW

Muon optimizer 在 ROCm 上会导致 NaN loss

In [ ]:
# 修改 gpt.py - 替换 Muon 为 AdamW
gpt_path = '/mnt/workspace/nanochat/nanochat/gpt.py'

with open(gpt_path, 'r') as f:
    content = f.read()

# 查找 Muon optimizer 代码块
old_muon = '''# Muon groups (matrix params, grouped by shape for stacking)'''
new_adamw = '''# AdamW groups for ALL params (modified for ROCm compatibility)'''

if old_muon in content:
    print('Found Muon optimizer code - needs manual patch')
    print('Please manually replace Muon with AdamW in gpt.py')
else:
    print('Code may already be patched or different version')

print(f'File: {gpt_path}')

---
## 6. 开始训练

In [ ]:
import subprocess
import os
os.chdir('/mnt/workspace/nanochat')

# 训练参数
train_cmd = [
    'python3', '-u', '-m', 'scripts.base_train',
    '--depth=12',
    '--max-seq-len=2048',
    '--device-batch-size=64',
    '--total-batch-size=262144',
    '--num-iterations=500',
    '--run=rocm_test',
    '--sample-every=-1',
    '--save-every=100',
    '--eval-every=50',
    '--window-pattern', 'L'
]

print('Starting training...')
print('Command:', ' '.join(train_cmd))
print('\nNote: This may take several minutes. For full training, increase num-iterations.')

# 运行训练 (前台，会阻塞)
# result = subprocess.run(train_cmd, capture_output=True, text=True, timeout=3600)
# print(result.stdout[-1000:])

# 或者后台运行
process = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(f'Training started (PID: {process.pid})')
print('Check progress with: rocm-smi --showuse')

In [ ]:
# 检查 GPU 使用情况
import subprocess
result = subprocess.run(['rocm-smi', '--showuse', '--showmeminfo', 'vram', '--showpower'], capture_output=True, text=True)
print(result.stdout)

---
## 7. 测试生成效果

In [ ]:
import os
os.environ['NANOCHAT_DTYPE'] = 'float32'

import torch
import sys
sys.path.insert(0, '/mnt/workspace/nanochat')

from nanochat.gpt import GPT, GPTConfig
from nanochat.tokenizer import get_tokenizer
import glob

print('Imports successful')

In [ ]:
# 加载最新 checkpoint
checkpoint_dir = os.path.expanduser('~/.cache/nanochat/base_checkpoints/d12/')
model_files = glob.glob(f"{checkpoint_dir}/model_*.pt")
model_files.sort(key=lambda x: int(x.split('model_')[1].split('.pt')[0]))

if model_files:
    checkpoint_path = model_files[-1]
    step = int(checkpoint_path.split('model_')[1].split('.pt')[0])
    
    print(f'Loading: {checkpoint_path}')
    print(f'Step: {step}')
    
    state_dict = torch.load(checkpoint_path, map_location='cuda', weights_only=True)
    print('Checkpoint loaded')
else:
    print('No checkpoints found. Run training first.')

In [ ]:
# 创建模型
config = GPTConfig(
    sequence_len=2048,
    vocab_size=32768,
    n_layer=12,
    n_head=6,
    n_kv_head=6,
    n_embd=768,
    window_pattern='L'
)

model = GPT(config)
if 'state_dict' in dir():
    model.load_state_dict(state_dict)
model = model.to('cuda').to(torch.float32)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {num_params/1e6:.1f}M')

In [ ]:
# 加载 tokenizer
tokenizer = get_tokenizer()
print(f'Tokenizer vocab size: {tokenizer.get_vocab_size()}')

In [ ]:
# 定义生成函数
def generate(prompt, max_tokens=50, temperature=0.7, top_k=40):
    encoded = tokenizer.encode(prompt)
    tokens = torch.tensor([encoded], device='cuda')
    
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(tokens[:, -2048:] if tokens.shape[1] > 2048 else tokens)
            logits = logits[:, -1, :]
            
            if temperature > 0:
                logits = logits / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
    
    return tokenizer.decode(tokens[0].tolist())

print('Generate function ready')

In [ ]:
# 测试生成
prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The most important thing in life is",
    "Python is a programming language that",
]

print('='*60)
print('Generation Test')
print('='*60)

for prompt in prompts:
    print(f'\n[Prompt]: {prompt}')
    for temp in [0.3, 0.7]:
        try:
            output = generate(prompt, max_tokens=30, temperature=temp)
            generated = output[len(prompt):].strip()[:80]
            print(f'[Temp {temp}]: {generated}')
        except Exception as e:
            print(f'[Temp {temp}]: Error - {e}')
    print('-'*60)

---
## 8. 结果分析

In [ ]:
# 分析训练结果
import json
import glob

meta_files = glob.glob(os.path.expanduser('~/.cache/nanochat/base_checkpoints/d12/meta_*.json'))
meta_files.sort()

if meta_files:
    print('Validation BPB History:')
    print('-'*40)
    for f in meta_files[-5:]:
        with open(f) as fp:
            data = json.load(fp)
            step = data.get('step', '?')
            val_bpb = data.get('val_bpb', '?')
            print(f'Step {step}: val_bpb = {val_bpb:.4f}' if isinstance(val_bpb, float) else f'Step {step}: val_bpb = {val_bpb}')
else:
    print('No meta files found')

In [ ]:
# 总结
print('='*60)
print('训练总结')
print('='*60)
print(f'模型: depth=12, params=286M')
print(f'环境: AMD MI300X, ROCm 6.10.5')
print(f'显存利用率: ~84% (172GB/192GB)')
print(f'Token/sec: ~35,650')
print('\n注意事项:')
print('1. Muon optimizer 在 ROCm 不稳定，需替换为 AdamW')
print('2. 使用 float32 避免 dtype 问题')
print('3. Tokens/Params ratio 需要 ~20 才能产生连贯文本')
print('4. 建议训练 5000+ steps')
print('='*60)